# IR ALÉM 1 — Extração de Informações Clínicas & Geração Diagnóstica com IA Generativa Real

Este notebook implementa a integração **real** entre o sistema conversacional do **CardioAI (Fase 5)**, o classificador de arritmias por CNN **(Fase 4)** e modelos fundacionais de ponta (**OpenAI `gpt-4o-mini`**) via Structured Outputs (`pydantic` + JSON schema).

### Objetivos do Módulo:
1. **Extração Estruturada:** Processar relatos clínicos não estruturados em linguagem natural e extrair sintomas, duração, intensidade, fatores de risco e nível de urgência;
2. **Correlação Multimodal:** Mapear os achados com as 5 classes de eletrocardiograma da AAMI (N, S, V, F, Q);
3. **Explicação Diagnóstica Humanizada:** Gerar laudos empáticos e orientações clínicas seguras para o paciente com base em seu histórico e no resultado do ECG;
4. **Resiliência e Governança:** Execução 100% ancorada em APIs reais de LLM. Sem chave de API (`OPENAI_API_KEY`), o notebook interrompe com erro explícito de governança, sem jamais simular dados médicos.

In [1]:
# ==============================================================================
# 1. Configuração de Ambiente, Dependências e Chaves de API (Colab / Local)
# ==============================================================================
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Tenta carregar .env local, nos diretórios pais ou raiz do CardioAI
caminho_notebook = Path('.').resolve()
for env_candidate in [
    caminho_notebook / '.env',
    caminho_notebook.parent / '.env',
    caminho_notebook.parent.parent / '.env',
    caminho_notebook.parent.parent.parent / '.env',
]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        break
else:
    load_dotenv()

# Suporte nativo a Google Colab Secrets (ícone de chave no Colab)
try:
    from google.colab import userdata
    if 'OPENAI_API_KEY' in userdata and not os.getenv('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

api_key = os.getenv('OPENAI_API_KEY')
if not api_key or not api_key.strip() or api_key.startswith('sua_chave'):
    raise RuntimeError(
        '❌ ERRO CRÍTICO DE CONFIGURAÇÃO: Chave OPENAI_API_KEY não configurada!\n'
        'Para executar a extração clínica real, configure sua chave no arquivo .env '
        'ou adicione aos Secrets do Google Colab (chave: OPENAI_API_KEY).\n'
        'O CardioAI opera estritamente com chamadas reais a modelos generativos de produção.'
    )

print(f'✓ OpenAI API Key configurada com sucesso: {api_key[:8]}...{api_key[-4:]}')
print(f"✓ Modelo Padrão: {os.getenv('OPENAI_MODEL', 'gpt-4o-mini')}")


✓ OpenAI API Key configurada com sucesso: sk-proj-...940A
✓ Modelo Padrão: gpt-4o-mini


In [2]:
# ==============================================================================
# 2. Importação do Módulo de Integração Real (integracao_llm_ecg.py)
# ==============================================================================
sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd().parent))

from integracao_llm_ecg import (
    extrair_informacoes_clinicas_real,
    gerar_explicacao_diagnostica_humanizada,
    ExtracaoClinica,
    CLASSES_ECG_FASE4
)

print('✓ Módulo integracao_llm_ecg importado com sucesso!')
print(f'✓ Schema estruturado Pydantic: {ExtracaoClinica.__name__}')
print(f'✓ Classes de ECG mapeadas (Fase 4): {list(CLASSES_ECG_FASE4.keys())}')


✓ Módulo integracao_llm_ecg importado com sucesso!
✓ Schema estruturado Pydantic: ExtracaoClinica
✓ Classes de ECG mapeadas (Fase 4): ['N', 'S', 'V', 'F', 'Q']


C:\Users\JonasLuisdaSilva\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Testes Práticos de Extração Clínica Estruturada (Chamadas Reais)

Abaixo, enviamos 3 relatos clínicos heterogêneos para a API da OpenAI. O modelo `gpt-4o-mini` analisa as queixas e extrai obrigatoriamente a estrutura Pydantic `ExtracaoClinica`:

In [3]:
# ==============================================================================
# 3. Execução das Extrações Reais com LLM
# ==============================================================================
casos_uso = [
    {
        'id': 1,
        'texto': 'Doutor, meu coração está muito acelerado faz mais de uma hora. Já tive episódio de arritmia antes e estou preocupado.'
    },
    {
        'id': 2,
        'texto': 'Estou com uma dor no peito muito forte, insuportável, irradiando para o braço esquerdo. Também sinto muita falta de ar. Sou fumante e hipertenso.'
    },
    {
        'id': 3,
        'texto': 'Gostaria de agendar meu retorno anual. Fiz o ECG de rotina semana passada e queria mostrar os resultados.'
    }
]

resultados_extracao = []

for caso in casos_uso:
    print('=' * 70)
    print(f"CASO {caso['id']}:")
    print(f"Mensagem do Paciente: \"{caso['texto']}\"")
    
    # Chamada REAL à OpenAI
    resultado = extrair_informacoes_clinicas_real(caso['texto'])
    
    if 'erro' in resultado:
        raise RuntimeError(f"Falha na extração clínica do Caso {caso['id']}: {resultado['erro']}")
        
    resultados_extracao.append(resultado)
    print('\n📦 Dados Extraídos pela LLM (JSON Estruturado):')
    print(json.dumps(resultado, indent=2, ensure_ascii=False))
    
    # Decisão de Integração no Assistente (Watson / Portal)
    print('\n🏥 Conexão com Sistema CardioAI:')
    if resultado.get('urgencia') == 'alta':
        print('🚨 AÇÃO WATSON / PORTAL: Acionar nó #emergencia -> SAMU 192 imediato!')
    elif resultado.get('classe_ecg_provavel') == 'S':
        print('💡 AÇÃO FASE 4: Provável arritmia Supraventricular (Classe S). Triagem com prioridade moderada.')
    else:
        print('✅ AÇÃO WATSON / PORTAL: Acionar nó #agendamento -> Fluxo ambulatorial padrão.')


CASO 1:
Mensagem do Paciente: "Doutor, meu coração está muito acelerado faz mais de uma hora. Já tive episódio de arritmia antes e estou preocupado."



📦 Dados Extraídos pela LLM (JSON Estruturado):
{
  "sintomas": [
    "coração acelerado",
    "episódio de arritmia"
  ],
  "duracao": "mais de uma hora",
  "intensidade": "moderada",
  "fatores_risco": [
    "arritmia prévia"
  ],
  "urgencia": "media",
  "classe_ecg_provavel": "S"
}

🏥 Conexão com Sistema CardioAI:
💡 AÇÃO FASE 4: Provável arritmia Supraventricular (Classe S). Triagem com prioridade moderada.
CASO 2:
Mensagem do Paciente: "Estou com uma dor no peito muito forte, insuportável, irradiando para o braço esquerdo. Também sinto muita falta de ar. Sou fumante e hipertenso."



📦 Dados Extraídos pela LLM (JSON Estruturado):
{
  "sintomas": [
    "dor no peito",
    "falta de ar"
  ],
  "duracao": "não informado",
  "intensidade": "alta",
  "fatores_risco": [
    "fumante",
    "hipertenso"
  ],
  "urgencia": "alta",
  "classe_ecg_provavel": "V"
}

🏥 Conexão com Sistema CardioAI:
🚨 AÇÃO WATSON / PORTAL: Acionar nó #emergencia -> SAMU 192 imediato!
CASO 3:
Mensagem do Paciente: "Gostaria de agendar meu retorno anual. Fiz o ECG de rotina semana passada e queria mostrar os resultados."



📦 Dados Extraídos pela LLM (JSON Estruturado):
{
  "sintomas": [],
  "duracao": "não informado",
  "intensidade": "baixa",
  "fatores_risco": [],
  "urgencia": "baixa",
  "classe_ecg_provavel": "N"
}

🏥 Conexão com Sistema CardioAI:
✅ AÇÃO WATSON / PORTAL: Acionar nó #agendamento -> Fluxo ambulatorial padrão.


## 4. Geração Diagnóstica Humanizada Cruzando Dados Clínicos e CNN de ECG (Fase 4)

Aqui demonstramos a união completa do ecossistema: o paciente relata sintomas no chat (Fase 5), o modelo de Deep Learning CNN (Fase 4) classifica o traçado eletrocardiográfico, e a LLM sintetiza uma comunicação acolhedora, precisa e personalizada.

> **Valores de exemplo representando uma saída da CNN da Fase 4 (o modelo .keras não é versionado por tamanho). Os dados do paciente são fictícios.**


In [4]:
# ==============================================================================
# 4. Geração de Explicação Humanizada com Análise Multimodal
# ==============================================================================
# Valores de exemplo representando uma saída da CNN da Fase 4 (o modelo .keras não é versionado por tamanho). Os dados do paciente são fictícios.
exemplo_saida_fase4 = {
    'classe_predita': 'V',
    'confianca': 0.964,
    'frequencia_cardiaca_bpm': 104,
    'ritmo': 'Ectopia Ventricular Frequente',
    'data_exame': '2026-10-15'
}

dados_paciente_exemplo = {
    'nome': 'Carlos Eduardo Silveira',
    'idade': 58,
    'sexo': 'Masculino',
    'historico': ['Hipertensão arterial sistêmica', 'Dislipidemia'],
    'sintomas_relatados': 'Sensação de aperto no peito após esforço físico moderado e tontura leve.'
}

# Valores de exemplo representando uma saída da CNN da Fase 4 (o modelo .keras não é versionado por tamanho). Os dados do paciente são fictícios.
dados_exame_utilizado = exemplo_saida_fase4

print('=' * 70)
print(f"Paciente: {dados_paciente_exemplo['nome']} ({dados_paciente_exemplo['idade']} anos)")
print(f"Achado Fase 4 CNN: Classe {dados_exame_utilizado['classe_predita']} ({dados_exame_utilizado['confianca']*100:.1f}% confiança)")
print('=' * 70)

laudo_humanizado = gerar_explicacao_diagnostica_humanizada(
    dados_paciente=dados_paciente_exemplo,
    dados_exame=dados_exame_utilizado
)

if not laudo_humanizado['sucesso']:
    raise RuntimeError(f"Erro ao gerar explicação humanizada: {laudo_humanizado['erro']}")

print('\n💬 Explicação Humanizada Gerada pela LLM:')
print('-' * 70)
print(laudo_humanizado['explicacao'])
print('-' * 70)
print(f"📊 Metadados: Modelo: {laudo_humanizado['modelo_utilizado']} | Consumo: {laudo_humanizado.get('tokens_utilizados')}")


Paciente: Carlos Eduardo Silveira (58 anos)
Achado Fase 4 CNN: Classe V (96.4% confiança)



💬 Explicação Humanizada Gerada pela LLM:
----------------------------------------------------------------------
Olá, Carlos Eduardo. Agradeço por compartilhar seus sintomas e por confiar em nós para entender melhor o que está acontecendo com seu coração. Vamos conversar sobre os resultados do seu eletrocardiograma (ECG) e o que eles significam para você.

O exame revelou que você apresenta uma condição chamada "ectopia ventricular frequente". Isso significa que há batimentos cardíacos que estão se originando de uma parte do coração chamada ventrículos, e esses batimentos estão ocorrendo de forma prematura. Essa situação pode ser um sinal de que o coração está passando por algum tipo de estresse, que pode estar relacionado à sua hipertensão arterial e dislipidemia, que são condições que você já conhece e que podem aumentar o risco de problemas cardíacos.

Além disso, você mencionou sentir um aperto no peito após esforço físico e uma leve tontura. Esses sintomas são importantes e podem 